# Menu Tracker
This notebook can run in both local (e.g. via docker container) and Colab hosted environment. It will automatically detect the environment and run the appropriate code.

## Three modes to run this notebook
1. **Colab + Docker**: Colab connected to local environment via docker container
  Colab connected to local environment via docker container (useful to work around the bot detection issue in Colab hosted environment)
1. **Colab-hosted**: Colab hosted environment with Google Drive mounted (recommended for Colab users)
2. **Full Local**: Run both Notebook and runtime in local environment

## Notes for Colab + Docker mode only

To use Colab with Docker, 
1. run this command in your terminal to start the container
```
docker run -p 127.0.0.1:9000:8080 us-docker.pkg.dev/colab-images/public/cpu-runtime
```
2. Copy-and-paste the  using token url provided in the terminal output to allow Colab to connect to the local runtime.
3. Continue running the notebook. The collected data will be stored in the local runtime.

**Notes**
As the results are stored in the local runtime, you will can use the script at the end of this notebook to download the results to your local machine. See 


## Initialize a Colab-managed container

These steps apply to both Colab+Docker and Colab+Hosted environments.

In Colab+Docker mode, Google Drive cannot be mounted. The notebook will automatically save data in the local runtime instead.

You can either skip the Google Drive mounting step or run it and ignore the mounting error.

### Mounting Google Drive
 If you are connecting runtime in local environment (e.g. docker container), you can skip this step or just let it fails and ignore the error message.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except NotImplementedError:
    print("Mounting Google Drive is not supported in this environment (likely a local runtime).")
    print("If running in Colab + Docker mode, this is expected. Data will be stored locally.")
except Exception as e:
    print(f"An unexpected error occurred while mounting Google Drive: {e}")

### Create directory in Google Drive
This step is required to create and navigate to the working directory `menutracker` in Google Drive. If you are running the notebook in your own environment, you can skip this step.

In [ ]:
%mkdir -p "/content/drive/MyDrive/menutracker"
%cd "/content/drive/MyDrive/menutracker"

### Backup previous MenuTracker working codes to a timestamped folder
This step is to retain a backup of previous working codes in case you need to refer back to them. If you are running the notebook in your own environment, you can skip this step. And if you think it's unnecessary to keep a backup, please remove any exsisting "Menu_Tracker" folder (if exists) and skip this step.

In [ ]:
import os
import datetime

# Define the current folder path
current_folder_path = "/content/drive/MyDrive/menutracker/Menu_Tracker"

# Check if the folder exists before attempting to rename
if os.path.exists(current_folder_path):
    datetime_str = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    new_folder_name = f"Menu_Tracker_{datetime_str}"
    new_folder_path = os.path.join(os.path.dirname(current_folder_path), new_folder_name)
    os.rename(current_folder_path, new_folder_path)
    print(f"Folder renamed to: {new_folder_path}")
else:
    print(f"Folder not found: {current_folder_path}. No renaming performed.")

### Cloning Menu Tracker repository to host machine
This step is to clone the Menu Tracker project from GitHub to the host machine. If you have cloned the repository and are running in local environment, you may skip this step.

In [ ]:
import os
user = 'intake24'
repo_name = 'Menu_Tracker'
cmd_string = 'git clone --single-branch --branch main https://github.com/{0}/{1}.git'.format(user, repo_name)
os.system(cmd_string)
assert os.path.exists(f"/content/drive/MyDrive/menutracker/{repo_name}"), "Incorrect Password or Repo Not Found, please try again"

### Install Chrome and related drivers

In [ ]:
# Run this if you are in a local Docker container to install Chrome and related drivers
!apt-get update
!apt-get install -y wget gnupg
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update
!apt-get install -y google-chrome-stable

# Verify installation
!google-chrome --version

# if (platform.machine() != 'arm64'):
#   print("initialise and import google_colab_selenium")
#   colab_webdriver = setup_driver_colab()
#   colab_webdriver.quit()

### Change to repository folder
It is the working directory for all subsequent steps

In [ ]:
%cd "/content/drive/MyDrive/menutracker/Menu_Tracker"

## Installing dependencies
This step is to install the required dependencies for the Menu Tracker project. If you are running in Colab-hosted or Docker environment, you need to run this step to install the dependencies in the Colab environment.

### Install requirements

In [ ]:
%pip install -r requirements.txt

### Diagnose and set logging level

Check and install required dependencies for the notebook to run

In [ ]:
# Diagnostic: Check Python environment
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Architecture:", sys.platform)
import platform
print("Machine:", platform.machine())
print("Platform:", platform.platform())

import logging

logging.basicConfig(
    level=logging.INFO,  # choose level
    format="%(asctime)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,  # important in notebooks to reset handlers
)
# Optional: quiet chatty libs
logging.getLogger("selenium").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("uc").setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

logger.debug("Testing: debug message.")
logger.info("Testing: info message.")
logger.warning("Testing: warning message.")
logger.error("Testing: error message.")
logger.critical("Testing: critical message.")

### Import dependencies and data destination
Import required dependencies for the notebook to run, and create the data directories if they do not already exist. I.e.
1. If it is detected as running on Colab, data will be stored at `/content/drive/MyDrive/menutracker/default_collecton`, s.t. can be retrieved via Google Drive
2. If it is running locally, data will be stored at `default_collection` or folder specified in the next step.

In [ ]:
from define_collection_wave import folder, create_collection
from helpers import combo_PDFDownload, combo_PDFDownload_class_name, RunScript, greene_king_download, combo_imgDownload, PDFDownloader, create_folder, selenium_PDF, setup_driver_colab

## Define collection wave
Define the collection wave by specifying the folder name.

If it is not specified, the default folder name will be `default_collection`.

In [ ]:
# Change next line to specify target folders. e.g. 

create_collection("Aug_collection_2026")

## Web scraping core scripts

### 1. McDonald's

In [ ]:
# RunScript('1_McDonalds')
%run food-chains/1_McDonalds.py

### 2. Wetherspoons

In [ ]:
# RunScript('2_Wetherspoons')
%run food-chains/2_Wetherspoons.py

### 3. Costa Coffee

20251121: Stronger bot detection prohibit the script to finish on Colab. Now it can only run successfully in local with VPN off.

20260209: Rewrite script to use Selenium instead of hardcoded GraphQL

20260223: Update script to use Selenium to web-scrape the website, ditched the GraphQL method

In [ ]:
# RunScript('3_CostaCoffee_selenium')
# %run food-chains/3_CostaCoffee.py
%run food-chains/3_CostaCoffee_selenium.py

### 4. Greggs

In [ ]:
# RunScript('4_Greggs')
%run food-chains/4_Greggs.py


### 5. KFC

Using *undetectedChrome* driver for Colab

In [ ]:
# Using *undetectedChrome* driver for Colab
# %run food-chains/5_KFC_colab.py

# RunScript('5_KFC')
%run food-chains/5_KFC.py

### 6. Domino's
20250918 - Need to run locally as Domino's has bot protection that Colab cannot bypass

20260210 - Pass running locally and on Colab without changes.

In [ ]:
# RunScript('6_Dominos')
%run food-chains/6_Dominos.py


### 7. Starbucks

In [ ]:
combo_PDFDownload('7_Starbucks', url='https://www.starbucks.co.uk/nutrition')
# RunScript('7_starbucks')

### 8. Pizzahut

In [ ]:
combo_PDFDownload(url='https://www.pizzahut.co.uk/restaurants/food/nutritional-information/', prex=
'https://www.pizzahut.co.uk', rest_name='8_Pizzahut')

### 9. Subway

20251121: Subway bot detection enhanced to become more sensitive of reused headers. Fixed with randomizing headers every time.

20260223: Cannot reproduce ticket (https://intake24.atlassian.net/browse/MENUTRACKR-65) on Colab or local environment. Possibly Colab server IP has been blocked at that time. Retry later may work.

In [ ]:
combo_PDFDownload(url='https://www.subway.com/en-gb/menunutrition/nutrition', prex=
'https://www.subway.com', rest_name='9_Subway')

### 10. Nando's

In [ ]:
# RunScript('10_Nandos')
%run food-chains/10_Nandos.py

### 11. Pizza Express

In [ ]:
combo_PDFDownload(rest_name='11_PizzaExpress', keyword='pdf',
                  url='https://www.pizzaexpress.com/allergens-and-nutritionals',
                  prex='https://www.pizzaexpress.com')

### 12. Burger King

20260225: Revised GraphQL to collect data with latest query parameters and structure

In [ ]:
# RunScript('12_BurgerKing')
%run food-chains/12_BurgerKing.py

### 13. Pret A Manger

In [ ]:
# RunSpider('13_Pret', folder)
%run food-chains/13_Pret.py

### 14. Caffe Nero -> requests API


In [ ]:
# RunScript('14_CaffeNero')
%run food-chains/14_CaffeNero.py


### 15. Wagamama -> Selenium

20260528: Wagamama fix to reflect website changes.

In [ ]:
# RunSpider('15_Wagamama', folder)
%run food-chains/15_Wagamama.py

### 16. Beefeater

20260209: Create script to collect data from web pages

In [ ]:
# combo_PDFDownload(url='https://www.beefeater.co.uk/en-gb/allergy-nutrition',
#                   rest_name='16_Beefeater', prex='https://www.beefeater.co.uk')
%run food-chains/16_Beefeater.py

### 17. Brewers Fayre

20260209: Create script to collect data from web pages (similar to 16_Beefeater)

20260910: Brewers Fayre has closed -- Whitbread folded it into Premier Inn dining. https://www.brewersfayre.co.uk/en-gb/allergy-nutrition now 301s to the homepage, titled "Brewers Fayre is now closed. Dining continues for Premier Inn Guests". No site left to scrape; script renamed food-chains/17_BrewersFayre_obsolete.py and removed from scraper_manifest.json.

In [ ]:
# combo_PDFDownload(url='https://www.brewersfayre.co.uk/en-gb/allergy-nutrition',
#                   rest_name='16_Beefeater', prex='https://www.brewersfayre.co.uk')
# 20260910: chain closed, see markdown note above
# %run 17_BrewersFayre.py


### 18. Sizzling Pubs
 20251028: Website seems to have stronger bot protection and cannot be accessed via undetectedChrome in Colab or local environment.
 
 20251028: Solved bot detection by using undetectedChrome, collecting data through its json-ld payload.

 20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

 20260609: Fixed by addressing new website menu layout by using revised CSS selector to look for menus

In [ ]:
# RunSpider('18_Sizzling', folder)
%run food-chains/18_Sizzling.py 

### 19. Ember Inns
Site moved, needs to be re-done, similar to Sizzling Pubs

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

In [ ]:
# RunSpider('19_EmberInns', folder)
%run food-chains/19_EmberInns.py

### 20. Chef & Brewer Pub Co.

Currently using siteid=6145 which is not working / irrelevant, and PDF is for Farmhouse Inns not Chef & Brewer.

20250910: Food menu found at  https://www.smartchef.co.uk/brands/ChefBrewer?siteid=6199

20251007: Menu download https://www.chefandbrewer.com/pubs/cambridgeshire/bridge/menu?type=main+menu&section=where+to+begin%3f

20251104: Updated to use new method to download food menu PDF from UI using Selenium.

In [ ]:
# greene_king_download(rest_name='20_Chef', id=6145, url='https://www.chefandbrewer.com/', folder=folder)
# RunSpider('20_ChefBrewer', folder)
# Download the PDF for kcal data
%run 'food-chains/20_ChefBrewer.py'

### 21. Table Table

20260910: Table Table has closed -- Whitbread folded it into Premier Inn dining. https://www.tabletable.co.uk/en-gb/allergy-nutrition now 301s to the homepage, titled "Table Table is now closed. Dining continues for Premier Inn Guests". No site left to scrape; script renamed food-chains/21_TableTable_obsolete.py and removed from scraper_manifest.json.

In [ ]:
# combo_PDFDownload(url='https://www.tabletable.co.uk/en-gb/allergy-nutrition', rest_name='21_TableTable',
#                   prex='https://www.tabletable.co.uk')

# 20260910: chain closed, see markdown note above
# %run '21_TableTable.py'

### 22. Toby Cavery
Site moved, needs to be re-done, similar to Sizzling Pubs

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

20260609: Updated the obsolete URL/selector to match the current Toby menu page. 

In [ ]:
# RunSpider('22_Toby', folder) 
%run food-chains/22_TobyCarvery.py

### 23. Revolution
Nutrition info not available on site, only allergen info available on https://book.revolution-bars.co.uk/allergens

20251111: Updated to use new method to extract allergen data from UI using Selenium.

*(Note: May require multiple attempts to run due to intricate UI interactions.)*

In [ ]:
# RunSpider('23_Revolution', folder)
# %run food-chains/23_Revolution.py
%run food-chains/23_Revolution_v2.py

### 24. Zizzi
20250910 - started providing calorie only on web pages

In [ ]:
# combo_PDFDownload('24_Zizzi', url='https://www.zizzi.co.uk/menus')
# RunSpider('24_Zizzi', folder)
%run food-chains/24_Zizzi.py

### 25. Ask Italian
20250910 - no nutrition info on site, only price and energy data on site, and allergen info available via PDF link in script tag

In [ ]:
# combo_PDFDownload(rest_name='25_Ask', url='https://www.askitalian.co.uk/allergens/')
# RunSpider('25_Ask', folder)
%run food-chains/25_Ask.py


### 26. Papa Johns - Nutrition calculators available only for US and Canada locations
The homepage has bot protection. Run the browser-native downloader locally; Colab may still be blocked.

In [ ]:
# combo_PDFDownload('26_PapaJohns', url='https://www.papajohns.co.uk/', prex='https://www.papajohns.co.uk')
selenium_PDF(
    '26_PapaJohns',
    url='https://www.papajohns.co.uk/allergens-and-nutrition',
    xpath_="//a[contains(@href, '/static/assets/pdfs/nutritional-information.pdf')]",
    prefix='https://www.papajohns.co.uk',
    download_via_browser=True,
    wait_time=5,
)

### 27. Yates
TBC: Ported, but I have some doubts on the source of data

20251113: Data collect seems to be correct

In [ ]:
# RunSpider('27_Yates', folder)
%run food-chains/27_Yates


### 28. Yo!Sushi -> Yo!Sushi and Yo!Sushi Tesco Kiosk
20250925 The data no longer presented as PDF, but a website hosted by tenkites.com. Scripts re-written to scrape from the website instead.
- https://menus.tenkites.com/yosushi/allergenpageyosushi
- https://menus.tenkites.com/yosushi/kiosk02


In [ ]:
# combo_PDFDownload('28_Yosushi', url='https://yosushi.com/legal/allergen-information',
#                   prex='https://yosushi.com')
%run food-chains/28_Yosushi.py
%run food-chains/28_Yosushi_Tesco.py


### 29. All Bar One

Site moved, needs to be re-done, similar to Sizzling Pubs

20260210: Fixed 3 months ago.

In [ ]:
# RunSpider('29_AllBarOne', folder)
%run food-chains/29_AllBarOne

### 30. GBK

In [ ]:
# RunScript('30_GBK')
%run food-chains/30_GBK


### 31. Flaming Grill -> PDF

In [ ]:
# RunScript('31_FlamingGrill')
%run food-chains/31_FlamingGrill

### 32. Loch Fyne seafood grill -> No nutrition available
This restaurant chain is closing down, and nutrition info is no longer available on their website.
https://www.greeneking.co.uk/pubs-restaurants-hotels/loch-fyne

In [ ]:
# combo_PDFDownload('32_LochFyne', prex='https://www.lochfyneseafoodandgrill.co.uk',
#                   url='https://www.lochfyneseafoodandgrill.co.uk/allergens')
# RunSpider('32_LochFyne', folder)
# combo_PDFDownload_class_name('32_LochFyne', url='https://www.lochfyneseafoodandgrill.co.uk/menu', keyword='menus-download')



### 33. PAUL

In [ ]:
# RunSpider('33_Paul', folder)
%run food-chains/33_Paul

### 34. Wimpy


In [ ]:
# RunSpider('34_Wimpy', folder)
%run food-chains/34_Wimpy

### 35. Krispy Creme
20250918 - The website employs IP check and ban VPN IPs, but no problem to run locally or on Colab

In [ ]:
# RunSpider('35_KrispyKreme', folder) -> not available yet, PDF
#combo_PDFDownload('35_KrispyKreme', url='https://www.krispykreme.co.uk/nutritionals')
# RunScript('35_krispyKreme', folder)

# java_PDF('35_KrispyKreme', url='https://www.krispykreme.co.uk/nutritionals', prex='https://www.krispykreme.co.uk',link_=False, xpath_="//*[@class='pagebuilder-button-primary']")

selenium_PDF('35_KrispyKreme', url='https://www.krispykreme.co.uk/nutritionals', prefix='https://www.krispykreme.co.uk', xpath_="//*[@class='pagebuilder-button-primary']")


### 36. Bill's
Scripts that extract data from tenkites

In [ ]:
# RunScript('36_Bills', folder)
%run food-chains/36_Bills

### 37. Walkabout

In [ ]:
# Walkabout
%run food-chains/37_Walkabout.py

### 38. Itsu

20260209: New script for web scraping

In [ ]:
# Itsu
%run food-chains/38_Itsu.py

### 39. Ben & Jerry
20250918: Need to run locally as Ben & Jerry's has bot protection that Colab cannot bypass

20260223: Added selenium version to workaround bot protection, but slower

In [ ]:
# RunScript('39_BenJerry')

# Seleium version to workaround bot protection, but slower
%run food-chains/39_BenJerry_selenium.py 
# %run food-chains/39_BenJerry.py


### 40. Asda -> PDF format
Randomly selected Asda - menu has not been available for a few rounds (20/08/2024)

20250912 - New location selected as previous one no longer has menu available

20250912 - New location has no menu available, added a check in the script to skip if no menu found

20250918 - Cannot find any cafe menu in Asda website

20260819 - Asda explicitly restricts nutrition data to local café menus. Supply an in-store export/photo.

In [ ]:
# combo_PDFDownload('40_Asda', url='https://storelocator.asda.com/east-of-england/stevenage/monkswood-way/cafe')
# combo_PDFDownload('40_Asda', url='https://storelocator.asda.com/london/london/151-east-ferry-road-isle-of-dogs/cafe')

### 41. Barburrito -> PDF


In [ ]:
# combo_PDFDownload(rest_name='41_Barburrito', url='https://www.barburrito.co.uk/menu')
# RunSpider('41_Barburrito', folder)
%run food-chains/41_Barburrito.py



### 42. Benugo

In [ ]:
# RunSpider('42_Benugo', folder)
%run food-chains/42_Benugo.py


### 43. Boost Juice

In [ ]:
# RunSpider('43_Boostjuice', folder)
%run food-chains/43_Boostjuice.py

### 44. Boswells

20260910: boswellsgroup.com's entire WordPress install is down (HTTP 500 on the homepage too, not just /menu/) -- "There has been a critical error on this website." External outage on their end, nothing to fix in our script. Left active in case the site recovers.

In [ ]:
combo_PDFDownload('44_Boswell', 'https://boswellsgroup.com/menu/')

### 45. Brewhouse


In [ ]:
combo_PDFDownload('45_Brewhouse', 'https://www.brewhouseandkitchen.com/menus/brewhouse-allergens')
# combo_PDFDownload('45_Brewhouse', 'https://www.brewhouseandkitchen.com/venue/bristol/')

### 46. Cineworld

20251121: Cineworld can run locally, but recently seems unable to run in Colab environment due to bot detection.

20260210: Still need to investigate the bot detection issue in Colab (please dowload manually and upload from https://www.cineworld.co.uk/static/en/uk/allergens-and-nutrition for now)

20260211: TODO Workaround bot detection issue in local environment by selenium_PDF, but doesn't work in Colab. May need to deploy in a dedicated server or use a different hosting service that can bypass bot detection, e.g. AWS EC2 instance with rotating proxies.

In [ ]:
selenium_PDF(
    '46_Cineworld',
    url='https://www.cineworld.co.uk/static/en/uk/allergens-and-nutrition',
    xpath_="//a[contains(@href,'jcr')]",
    prefix='https://www.cineworld.co.uk'
)


### 47. Coffee #1
20250912 - Revised URL as previous one no longer has PDF available

In [ ]:
# combo_PDFDownload('47_Coffee1', 'https://www.coffee1.co.uk/food-nutritional-information/')
combo_PDFDownload('47_Coffee1', 'https://www.coffee1.co.uk/allergy-advice/')


### 48. Common Rooms
leave as has not run for a few rounds, chain closed? (20/08/2024)

20250912 - Seems that the chain no longer provides nutrition info on their website


In [ ]:
#RunSpider('48_CommonRooms', folder)

### 49. Cookhouse & Pub

20260910: Cookhouse + Pub has closed -- Whitbread folded it into Premier Inn dining. https://www.cookhouseandpub.co.uk/en-gb/allergy-nutrition now 301s to the homepage, titled "Cookhouse + Pub is now closed. Dining continues for Premier Inn Guests". No site left to scrape; script renamed food-chains/49_CookhousePub_obsolete.py and removed from scraper_manifest.json.

In [ ]:
# 20260910: chain closed, see markdown note above
# combo_PDFDownload('49_CookhousePub', url='https://www.cookhouseandpub.co.uk/en-gb/',
#                   prex='https://www.cookhouseandpub.co.uk', verify=True)

### 50.Crussh -> terrible website!
20250912 - Seems that the website is no longer functioning properly, unable to access menu page

20251104: Site down, unable to access website

In [ ]:
# RunSpider('50_Crussh',json_ = True, folder = folder)
# %run 50_Crussh.py

### 51. Farmhouse Inns -> PDF

In [ ]:
# greene_king_download(rest_name='51_FarmhouseInns', id='5690', url='https://www.farmhouseinns.co.uk', folder=folder)
# RunScript('51_FarmhouseInns')
%run food-chains/51_FarmhouseInns.py

<cell_type>markdown</cell_type>### 52. Five guys -> PDF

20260820: Moved to a standalone script that finds the current guide by its stable button text ("UK Nutrition & Allergen Guide") instead of a filename keyword, since the PDF filename changes with every product update.

In [ ]:
# combo_PDFDownload(rest_name='52_FiveGuys', url='https://www.fiveguys.co.uk/menu', keyword = 'nutrition')
%run food-chains/52_FiveGuys.py

### 53. Harvester
20250912: Site moved, needs to be re-done, similar to Sizzling Pubs

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

20260609: Fixed Harvester using Sizzling’s updated menu selector pattern.

In [ ]:
# RunSpider('53_Harvester', folder)
%run food-chains/53_Harvester.py


### 54. Hungry Horse -> a greene king company

In [ ]:
# greene_king_download('54_HungryHorse', id='6347', url='https://www.hungryhorse.co.uk', folder=folder)
# RunScript('54_HungryHorse')
%run food-chains/54_HungryHorse.py

### 55. Joe & the Juice
20250918 - Need to run locally as it has bot protection that Colab cannot bypass, causing 403 forbidden error
20251007 - IP check seems to be implemented, need to disable VPN / proxy to run locally

In [ ]:
# RunSpider('55_JoeJuice', folder, json_=True)
# RunScript('55_JoeJuice')
%run food-chains/55_JoeJuice.py


### 56. Leon

In [ ]:
# RunSpider('56_Leon', folder)
%run food-chains/56_Leon.py

### 57. greene king

In [ ]:
# greene_king_download('57_GreeneKing', id='8183', url='https://www.greeneking-pubs.co.uk', folder=folder)
# RunScript('57_GreeneKing')
%run food-chains/57_GreeneKing.py

### 58. Vue -> PDF

20251113: PDF download seems to be working correctly, but sometimes strong bot protection will interfere web scraping. Advise to download PDFs manually at https://www.myvue.com/legal/nutritional-information if issues persist.

20251113: Wrote an new script to download PDFs

20251113: Update: still not helping due to bot protection. Advise to download PDFs manually at https://www.myvue.com/legal/nutritional-information if issues persist.


In [ ]:
# vue_PDF('58_Vue', url='https://www.myvue.com/legal/nutritional-information', 
#  xpath_="//a[contains(@href, 'media')]")

selenium_PDF('58_Vue', url='https://www.myvue.com/legal/nutritional-information', 
 xpath_="//a[contains(@href, 'media')]")

# %run food-chains/58_Vue.py

<cell_type>markdown</cell_type>### 59. Odeon Cinema

20250917: Apparently stronger bot checking stop PDF downloading, enhancements is needed.

20251007: Using undetected-chrome with fakeuseragent solved the problem. Sometime it still fails, probably due to IP check.

20251113: PDF download seems to be working correctly using undetectedChrome in local environment. May need to run locally or download PDFs manually at https://www.odeon.co.uk/experiences/food-drinks/food-and-drinks-facts-and-figures if issues persist.

20260820: The page now sits behind a queue-it waiting-room wall (redirects to odeon.queue-it.net); a real Selenium session still gets past it. Moved to a standalone script that filters for the current `-uk-version.pdf` files (nutritional information, allergens matrix, pre-packed product matrix), skipping the Belfast/ROI variants and an unrelated Costa PDF also linked on the same page.

In [ ]:
# java_PDF('59_Odeon', url='https://www.odeon.co.uk/experiences/food-drinks/food-and-drinks-facts-and-figures/',prex='https://www.odeon.co.uk',link_=False, xpath_="//p/a[contains(@title, 'Nutritional')]")
# selenium_PDF('59_Odeon', url='https://www.odeon.co.uk/experiences/food-drinks/food-and-drinks-facts-and-figures/',prefix='https://www.odeon.co.uk', xpath_="//p/a[contains(@title, 'Nutritional')]")
%run food-chains/59_Odeon.py

### 60. Marston's Pubs

In [ ]:
# combo_PDFDownload('60_Marstons', url='https://www.dragonflypubbasingstoke.co.uk/menus/')
# RunSpider('60_Marstons', folder)
%run food-chains/60_Marstons.py

### 61. Morrisons Cafe

In [ ]:
# RunScript('61_MorrisonsCafe')
%run food-chains/61_MorrisonsCafe.py

<cell_type>markdown</cell_type>### 62. Pho Cafe

20260820: Moved to a standalone script using the `/nutrition/` page with a `Guide` keyword filter, so it downloads only the allergen and nutritional-guideline PDFs instead of every PDF linked from the menus page (main menu, kids menu, etc.).

In [ ]:
# combo_PDFDownload('62_Pho', url='https://www.phocafe.co.uk/menus/', prex='https://www.phocafe.co.uk')
%run food-chains/62_Pho.py


### 63. Pieminister
20250916 - Script rewritten for new source website structure

In [ ]:
# RunSpider('63_Pieminister', folder)
%run food-chains/63_Pieminister.py


### 64. Pure

In [ ]:
# RunScript('64_Pure')
%run food-chains/64_Pure.py


### 65. Sainsbury Cafe

20250916 - Sainsbury closed all cafes in 2025

In [ ]:
# combo_PDFDownload('65_SainsburysCafe',
#                   'https://www.sainsburys.co.uk/shop/gb/groceries/get-ideas/our-instore-services/--sainsburys-cafe',
#                   prex='https://www.sainsburys.co.uk')
# combo_PDFDownload('65_SainsburysCafe',
#                   'https://help.sainsburys.co.uk/help/terms-and-conditions/sainsburyscafe',
#                   prex='https://www.sainsburys.co.uk')


### 66. Soho Cafe
20250916 - Source replaced by PDF link, script rewritten

In [ ]:
# RunSpider('66_SohoCafe', folder, json_=True)
combo_PDFDownload('66_SohoCafe', url='https://sohocoffee.com/allergens/', prex='https://sohocoffee.com')

### 67. Stonehouse Pizza
20250916: Site moved to https://www.stonehouserestaurants.co.uk/food#/ and need new script, similar to Sizzling Pubs

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.


In [ ]:
# RunSpider('67_StonehousePizza', folder)
%run food-chains/67_StonehousePizza.py

### 68. Tank and Paddle
20250916 - Similar to Walkabout using tkmenus.com as a base, script rewritten

In [ ]:
# RunSpider('68_TankPaddle', folder)
%run food-chains/68_TankPaddle.py

<cell_type>markdown</cell_type>### 69. Tesco Cafe
20250916 - Tesco seems to have provided allergen info as PDF and only calorie info on web pages, script rewritten

20260820: tesco.com blocks plain `requests` traffic with a persistent 403 (confirmed bot-fingerprint detection, not a rate limit); a real Selenium session loads normally. Script now fetches through Selenium and downloads the current GB allergen matrix PDF itself, so the separate `combo_PDFDownload` call below is gone (it always 403'd silently and never produced anything).

In [ ]:
# RunSpider('69_TescoCafe', folder)
%run food-chains/69_TescoCafe.py

### 70. The Cornish Bakery
20250916 - Download allergen PDF and calorie info from web pages


In [ ]:

# RunSpider('70_Cornish', folder)
%run food-chains/70_Cornish.py
combo_PDFDownload('70_Cornish', url='https://thecornishbakery.com/products/', prex='https://cdn.shopify.com/')


### 71. Thomas the Baker

In [ ]:
# RunSpider('71_ThomasBaker', folder)
%run food-chains/71_ThomasBaker.py

### 72. Tim Hortons

20260304: Enhanced script to collect all sizes of drinks

In [ ]:
# RunSpider('72_TimHortons', folder, json_=True)
%run food-chains/72_TimHortons.py

### 73. Top Golf
20250916 - Parsed nutrition info from web page instead of PDF

20260609 - Updated script to also download the PDF while compiling CSV data

In [ ]:
# tg_path = create_folder('73_TopGolf', folder)
# PDFDownloader(url='https://s3.topgolf.com/uploads/pdf/menus/topgolf-nutritional-information.pdf?v=20200131',
#               filePath=tg_path + '/top-golf-nutritional-information.pdf')
# RunSpider('73_TopGolf', folder)
# java_PDF(rest_name='73_TopGolf', url ='https://topgolf.com/uk/chigwell/menu/', link_=False, xpath_='//a[contains(@href, "topgolf.kitchencut.com")]')

%run food-chains/73_TopGolf.py

### 74. Town, Kitchen, and Pubs
20250916: Cannot find the website

20251104: Site down, unable to access website

In [ ]:
# RunSpider('74_TownKitchenPubs', folder)
# %run 74_TownKitchenPubs.py

### 75. Vintage Inns
20250916: Site moved to https://www.vintageinn.co.uk/food#/, needs to be re-done, similar to Sizzling Pubs

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

In [ ]:
# RunSpider('75_VintageInns', folder)
%run food-chains/75_VintageInns.py

<cell_type>markdown</cell_type>### 76. Wasabi

20260820: The old pinned 2024 PDF URL 404'd. Rewrote as a standalone script using `combo_PDFDownload` against the stable `/menus/` page, which correctly discovers the current nutritional guide PDF.

In [ ]:
# java_PDF('76_Wasabi', url = 'https://www.wasabi.uk.com/our-food/', link_=False, xpath_ = '//a[contains(@href, "nutrition")]')
#Folder_path = create_folder('76_Wasabi', define_collection_wave.folder)
#PDFDownloader(url='https://wasabiuk.wpengine.com/wp-content/uploads/2023/11/WAS_Nutritional_Guide_091123_V5.pdf',filePath= Folder_path +'/wasabi_nutrition.pdf')
# RunScript('76_Wasabi')

# 20250916 - Using more robust PDF download function
# combo_PDFDownload('76_Wasabi', url='https://www.wasabi.uk.com/menus/', prex='https://www.wasabi.uk.com')
%run food-chains/76_Wasabi.py

### 77. Waterfields - leave as has not run for a few rounds, chain closed? (20/08/2024)
20250916 - Seems that the chain no longer provides nutrition info on their website

In [ ]:
# RunSpider('77_Waterfields', folder, json_=True)

### 78. Birds Bakery


In [ ]:
# RunSpider('78_BirdsBakery', folder, json_=True)
# RunSpider('78_BirdsBakery', folder)
%run food-chains/78_BirdsBakery.py

### 79. Tortilla
20250916 - Added PDF download to complement relative unstable script due to complex interaction with source website.

In [ ]:
# RunScript('79_Tortilla')
%run food-chains/79_Tortilla.py
combo_PDFDownload('79_Tortilla', url='https://www.tortilla.co.uk/menu/nutrition-and-allergens', prex='https://www.tortilla.co.uk')

### 80. Tossed

20260910: Fixed selector drift from a site redesign (ordering platform vmos.io reworked its markup). The item "more details" trigger lost its `aria-label='More details.'`; the same icon is now an unlabelled button wrapping an inline svg tagged `data-test="Info"`. The nutrition modal container changed from `.ReactModal__Content` to `[data-test='modal-content']`, and the close button from `aria-label^='Close'` to `[data-test='close-modal-button']` (the `allergens-text` id and `meal-tab-2` id used deeper in extraction are unchanged). The redesigned modal also no longer shows the item name anywhere in its own DOM, so the old h1 lookup always fell back to "Unknown Item" -- now captured from the card's `[data-test='item-name']` before the modal opens. Verified end-to-end: 118 real records with full nutrition data and correct item names.

In [ ]:
# RunSpider('80_Tossed', folder)
# RunScript('80_tossed')
%run food-chains/80_tossed.py


### 81. Bella Italian
20250916 - Ported, but original approach to acquire JSON payload no longer work and need a overhaul. Allergen information is in https://viewthe.menu/8azv, need a script too.

20250923 - Script added, extracted PDF file from https://menus.tenkites.com/thebigtg/mobilemenus11

In [ ]:
# RunScript('81_BellaItalian')
%run food-chains/81_BellaItalian.py
selenium_PDF(rest_name='81_BellaItalian', url='https://menus.tenkites.com/thebigtg/mobilemenus11', use_partial_link_text=True, partial_link_value='DOWNLOAD ALLERGEN', handle_runtime_pdf=True, download_filename='bella_allergen')

### 82. Cafe Rouge
20250916 - Similar to 81_BellaItalian. Allergen info can be found in a PDF with link generated in run-time, need some handling.

20250923 - Script added, extracted PDF file from https://www.caferouge.com/restaurants/Center-Parcs/longleat/menu

20250924 - Revised script to handle new PDF link format

In [ ]:
# RunSpider('82_CafeRouge', folder)
%run food-chains/82_CafeRouge.py
selenium_PDF(rest_name='82_CafeRouge', url='https://www.caferouge.com/restaurants/Center-Parcs/longleat/menu', use_partial_link_text=True, partial_link_value='DOWNLOAD ALLERGEN', handle_runtime_pdf=True, download_filename='cafe_rouge_allergen')


### 83. Taco Bell

In [ ]:
# RunSpider('83_TacoBell', folder)
%run food-chains/83_TacoBell.py

### 84. Coco di mama -> no NI anymore

In [ ]:
# RunSpider('84_Coco',folder)
%run food-chains/84_Coco.py

### 85. The real greek
20250917 - Keep hardcoded menu items in updated script, and added PDF download script.

In [ ]:
# RunSpider('85_RealGreek', folder)
%run food-chains/85_RealGreek.py
combo_PDFDownload('85_RealGreek', url='https://www.therealgreek.com/menu/', prex='https://www.therealgreek.com')


### 86. Honest Burger
20250917 - NI and AI moved to https://menus.tenkites.com/honestburgers/honestburgers and need new script

20260923 - Script rewritten for new source website structure

In [ ]:
# combo_PDFDownload('86_HonestBurger', url='https://www.honestburgers.co.uk/allergy-information/', keyword='nutritional',
#                   prex='https://www.honestburgers.co.uk/')
%run food-chains/86_HonestBurger.py

<cell_type>markdown</cell_type>### 87. AMT

20260820: URL 404s. Confirmed via the menu page, homepage, and FAQ that amtcoffee.co.uk publishes no allergen/nutrition PDF, table, or image at all — the site says outright to ask in-store. Left as a retired chain, same as 40 Asda and 50 Crussh, until a digital source exists.

In [ ]:
# combo_PDFDownload('87_AMT', url='http://amtcoffee.co.uk/types/drinks/')

<cell_type>markdown</cell_type>### 88. Chicken Cottage

20260820: Allergens are published as a single chart image, not a PDF or HTML table. Moved to a standalone script that finds the current chart on the `/allergens/` page (not `/our-food/`, which is just food photography) and OCRs it into a JSON sidecar when the optional system `tesseract` binary is installed.

In [ ]:
# combo_imgDownload('88_ChickenCottage','https://chickencottage.com/our-food/',folder)
# combo_imgDownload('88_ChickenCottage','https://chickencottage.com/our-food/', folder)
%run food-chains/88_ChickenCottage.py

### 89. Browns
20250917: smartchef.co.uk website, need a new script.

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

In [ ]:
# RunSpider('89_Browns', folder)
%run food-chains/89_Browns.py

### 90. ONeills
20250917: smartchef.co.uk website, need a new script.

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

In [ ]:
# RunSpider('90_ONeills', folder)
%run food-chains/90_ONeills.py

### 91. Nicholson's
20250917: smartchef.co.uk website, need a new script, similar to Sizzlings

20251104: Updated to use new method to extract allergen and nutritional data from UI using Selenium.

In [ ]:
# RunSpider('91_Nicholsons', folder)
%run food-chains/91_Nicholsons.py

## Download collected data from local runtime (for Colab + Docker mode)

As the data is stored in the local runtime instead of google drive, you can use the following script to download the collected data to your local machine.


In [ ]:
from google.colab import files

# Create a timestamp string (e.g., 20260508_2030)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"collection_{timestamp}.zip"
zip_path = os.path.join("/content", zip_filename)

# ZIP the collection folder
!zip -r "{zip_path}" "{folder}"

# Download the resulting file
if os.path.exists(zip_path):
    files.download(zip_path)
else:
    print(f"Error: {zip_path} was not created. Check if the 'folder' variable is defined correctly.")

## Parallel runner (optional)

You can use the helper below to run scripts in parallel: it launches each scraper as a separate Python subprocess and controls concurrency with a thread pool.

- `max_workers=5` means up to 5 scripts run at the same time.
- Start with 2-3 workers for Selenium-heavy scripts (Chrome/anti-bot limits).
- Keep scripts writing to separate output folders (your numbered scripts already do this).

In [ ]:
from run_parallel import run_scripts_parallel

# Select a small batch from scraper_manifest.json (up to 5 at once)
# Tip: keep Selenium-heavy jobs in a smaller batch (2-3 workers) for stability.

scripts_to_run = [
    "22_TobyCarvery.py",
    "53_Harvester.py",
    "73_TopGolf.py",
]

results = run_scripts_parallel(scripts_to_run, max_workers=5)

# Optional: inspect full logs from one script
print(results["53_Harvester.py"]["stdout"])
print(results["53_Harvester.py"]["stderr"])
print(results["22_TobyCarvery.py"]["stdout"])
print(results["22_TobyCarvery.py"]["stderr"])
print(results["73_TopGolf.py"]["stdout"])
print(results["73_TopGolf.py"]["stderr"])
